# Gravica Tutorial: Schwarzschild Metric

**Gravica** is a symbolic general relativity library built on [Symbolica](https://symbolica.io).

In this tutorial we walk through the full computation pipeline using the Schwarzschild metric:

$$
g_{ab} \;\xrightarrow{\partial}\; \Gamma^a_{bc} \;\xrightarrow{\partial}\; R^a{}_{bcd} \;\xrightarrow{\text{trace}}\; R_{ab} \;\xrightarrow{-\frac{1}{2}g_{ab}R}\; G_{ab}
$$

Each stage uses the corresponding class: **MetricTensor → ChristoffelSymbols → RiemannTensor → RicciTensor → EinsteinTensor**.

In [1]:
import os
from symbolica import set_license_key

_key = os.environ.get("SYMBOLICA_LICENSE", "")
if _key:
    set_license_key(_key)

from gravica import (
    ChristoffelSymbols,
    EinsteinTensor,
    MetricTensor,
    RicciTensor,
    RiemannTensor,
    ricci_scalar,
    display,
    check,
)
from gravica.simplify import simplify, str_is_zero

The cell below runs the **full computation pipeline** — from metric to Einstein tensor in six lines.

In [ ]:
from symbolica import S, Expression, PrintMode


def _greek(name):
    tex = f"\\{name}"

    def _print(expr, mode, **kw):
        if mode == PrintMode.Latex:
            return tex

    return S(name, custom_print=_print)


t, r = S("t"), S("r")
theta = _greek("theta")
phi = _greek("phi")
r_s = S("r_s")
sin = S("sin")

ZERO = Expression.num(0)

g = MetricTensor(
    [
        [(r - r_s) / r, ZERO, ZERO, ZERO],
        [ZERO, -r / (r - r_s), ZERO, ZERO],
        [ZERO, ZERO, -(r**2), ZERO],
        [ZERO, ZERO, ZERO, -(r**2) * sin(theta) ** 2],
    ],
    (t, r, theta, phi),
)

coords = ["t", "r", "\\theta", "\\varphi"]

christoffel = ChristoffelSymbols(g)
riemann = RiemannTensor(christoffel)
ricci = RicciTensor(riemann)
einstein = EinsteinTensor(ricci)

## 1. Metric Tensor $g_{ab}$

The Schwarzschild metric describes the spacetime outside a spherically symmetric, static mass $M$.
In coordinates $(t, r, \theta, \varphi)$:

$$
ds^2 = \frac{r - r_s}{r}\,dt^2 - \frac{r}{r - r_s}\,dr^2 - r^2\,d\theta^2 - r^2\sin^2\theta\,d\varphi^2
$$

where $r_s = 2GM/c^2$ is the Schwarzschild radius.

Above, we built the metric by hand using `MetricTensor(components, coords)`. This approach works for any metric — just supply a 2D list of `symbolica.Expression` components and a tuple of coordinate symbols.

In [ ]:
print(f"Coordinates: {g.coords}")
print(f"Dimension: {g.dim}")
print(f"Diagonal: {g.is_diagonal()}")
print()
print("Metric components g_ab:")
display.components_table(display.nonzero_components(g, coords), g)

In [ ]:
inv_items = []
for i in range(g.dim):
    val = g.inverse[i][i]
    if not str_is_zero(val):
        inv_items.append((f"{coords[i]} {coords[i]}", val))

print("Inverse metric g^ab:")
display.components_table(inv_items, tensor_symbol="g", index_style="super")

## 2. Christoffel Symbols $\Gamma^a{}_{bc}$

The Christoffel symbols (second kind) are the components of the Levi-Civita connection:

$$
\Gamma^a{}_{bc} = \frac{1}{2} g^{ad}\left(\partial_b g_{dc} + \partial_c g_{bd} - \partial_d g_{bc}\right)
$$

In [ ]:
display.components_table(display.nonzero_components(christoffel, coords), christoffel)

## 3. Riemann Tensor $R^a{}_{bcd}$

The Riemann curvature tensor fully describes the curvature of spacetime:

$$
R^a{}_{bcd} = \partial_c \Gamma^a{}_{bd} - \partial_d \Gamma^a{}_{bc} + \Gamma^a{}_{ce}\Gamma^e{}_{bd} - \Gamma^a{}_{de}\Gamma^e{}_{bc}
$$

In [ ]:
riemann_items = display.nonzero_components(riemann, coords)

print(f"Non-zero independent components (c < d): {len(riemann_items)}")
display.components_table(riemann_items, riemann)

In [ ]:
from IPython.display import Markdown

antisym_ok = check.antisymmetry(riemann, simplify)
status = "All verified" if antisym_ok else "FAILED"
Markdown(f"**Antisymmetry check** $R^a_{{bcd}} + R^a_{{bdc}} = 0$: **{status}**")

## 4. Ricci Tensor $R_{ab}$ and Ricci Scalar $R$

The Ricci tensor is the trace of the Riemann tensor:

$$
R_{ab} = R^c{}_{acb}
$$

The Ricci scalar is the further trace:

$$
R = g^{ab} R_{ab}
$$

Since the Schwarzschild solution is a **vacuum solution** ($T_{ab} = 0$), we expect $R_{ab} = 0$ and $R = 0$.

In [ ]:
from IPython.display import Markdown

ricci_is_zero = check.zero(ricci, simplify)

R = simplify(ricci_scalar(ricci))
scalar_is_zero = str_is_zero(R)

print(f"Ricci scalar R = {R}")
vacuum = ricci_is_zero and scalar_is_zero
Markdown(
    f"**Vacuum solution check** ($R_{{ab}} = 0$, $R = 0$): **{'Verified' if vacuum else 'FAILED'}**"
)

## 5. Einstein Tensor $G_{ab}$

The Einstein tensor appears on the left-hand side of the Einstein field equations:

$$
G_{ab} = R_{ab} - \frac{1}{2} g_{ab} R
$$

For a vacuum solution, the Einstein equations $G_{ab} = 0$ must hold.

In [ ]:
from IPython.display import Markdown

einstein_is_zero = check.zero(einstein, simplify)

Markdown(
    f"**Vacuum Einstein equations** $G_{{ab}} = 0$: **{'Verified' if einstein_is_zero else 'FAILED'}**"
)